# Multimodal Deception Detection: Environment Setup

First, you must install OpenFace to extract facial Action Units and head pose data.
Follow the steps from [here](https://github.com/TadasBaltrusaitis/OpenFace/wiki#installation).
Afterwards, in the configuration section you must pinpoint to openface build binary feature extractor

# Imports and configuration

In [ ]:
import sys
import os
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"
import glob
import tempfile
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from src.extractors.visual.visual_feature_extractor import VisualFeatureExtractor
from huggingface_hub import snapshot_download
from scenedetect import detect, ContentDetector
from src.common.constants import OPENFACE_BIN
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# Hardware Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Directory & Binary Path Configurations
DATA_DIR = os.path.expanduser("~/visual_test/data/DOLO")
HF_DATASET_NAME = "dandyidk/Multi-Modal-Lie-Detection"
print(f"CUDA Available: {torch.cuda.is_available()} | Device: {DEVICE}")
print(f"Loading Hugging Face Dataset: {HF_DATASET_NAME}")
print(f"OpenFace Executable: {OPENFACE_BIN}")

# Remote Dataset Inspection

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

files = api.list_repo_files(
    repo_id="dandyidk/Multimodal",
    repo_type="dataset",
)

# Use a tuple of folder strings to check multiple paths
target_folders = ("DOLOS/Data/", "Real-life_Deception_Detection_2016/Clips") 

filtered_files = []
for filename in files:
    if filename.startswith(target_folders):
        filtered_files.append(filename)
        # pass 

print(f"Found {len(filtered_files)} files across the target folders.")

### Dataset Download and Metadata Extraction
Download the dataset using `snapshot_download`. Once downloaded, recursively locate all `.mp4` files and use OpenCV (`cv2`) to extract baseline video metadata, such as frame count, FPS, and total duration in seconds. This information is saved into a Pandas DataFrame (`meta_df`) for downstream processing.

In [ ]:
repo_dir = DATA_DIR
assert os.path.isdir(repo_dir), f"Dataset not found: {repo_dir}"
print(f"Using existing local dataset: {repo_dir}")

# Locate all local MP4 files recursively
video_paths = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.mp4"), recursive=True))

video_stats = []

for idx, video_path in enumerate(video_paths):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Warning: Failed to open video at {video_path}")
        continue

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    duration = frame_count / fps if fps and fps > 0 else 0.0

    video_stats.append({
        "sample_index": idx,
        "filename": os.path.basename(video_path),
        "path": video_path,  # Physical disk path for OpenFace
        "frame_count": frame_count,
        "fps": round(float(fps), 2),
        "duration_sec": round(float(duration), 2),
    })
    cap.release()

meta_df = pd.DataFrame(video_stats)
print(f"Successfully loaded and verified {len(meta_df)} video files.")
meta_df.head()

## EDA: el l3b kolo :D
this analysis evaluates a sliding window chunking approach, cutting long videos into multiple smaller, fixed-length segments (e.g., 8 seconds) to maintain a uniform tensor shape without excessive padding.

In [ ]:
# Ensure meta_df exists and has 'duration_sec' and 'fps' columns populated from earlier steps
if not meta_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot 1: The Bimodal Distribution
    sns.histplot(meta_df["duration_sec"], bins=40, kde=True, ax=axes[0], color="teal")
    axes[0].set_title("Video Duration Distribution (Bimodal)")
    axes[0].set_xlabel("Duration (seconds)")
    axes[0].set_ylabel("Number of Videos")
    
    # Chunking Parameters
    target_fps = int(meta_df["fps"].mode()[0])
    chunk_length_sec = 8.0  # Set to match the peak of the shorter dataset
    overlap_sec = 2.0       # Overlap to preserve context across boundaries
    
    chunk_frames = int(chunk_length_sec * target_fps)
    
    # Calculate how many chunks each video will produce
    def calculate_chunks(duration):
        if duration <= chunk_length_sec:
            return 1
        else:
            # Formula for sliding window chunks
            return int(np.ceil((duration - chunk_length_sec) / (chunk_length_sec - overlap_sec))) + 1

    meta_df['projected_chunks'] = meta_df['duration_sec'].apply(calculate_chunks)
    
    # Plot 2: Projected Dataset Size After Chunking
    sns.countplot(data=meta_df, x='projected_chunks', ax=axes[1], palette="magma")
    axes[1].set_title(f"Projected Chunks per Video (Window: {chunk_length_sec}s, Overlap: {overlap_sec}s)")
    axes[1].set_xlabel("Number of Chunks Generated")
    axes[1].set_ylabel("Frequency")

    plt.tight_layout()
    plt.show()

    total_original = len(meta_df)
    total_chunks = meta_df['projected_chunks'].sum()
    
    print("--- Chunking Strategy Recommendations ---")
    print(f"Target FPS              : {target_fps}")
    print(f"Sequence Length (T)     : {chunk_frames} frames ({chunk_length_sec} seconds)")
    print(f"Original Video Count    : {total_original}")
    print(f"Projected Dataset Size  : {total_chunks} chunked sequences")
    print("\nBy chunking, the 30-second clips act as data augmentation (providing multiple samples), ")
    print("and you avoid VRAM exhaustion during the cross-modal attention and GRU operations.")

### Deception Class Balance

In [ ]:
if not meta_df.empty:
    def extract_label(filename):
        fname_lower = filename.lower()
        if 'lie' in fname_lower or 'deception' in fname_lower:
            return 'Lie'
        elif 'truth' in fname_lower or 'true' in fname_lower:
            return 'Truth'
        else:
            print("stuff")
            return 'Unknown'
            
    meta_df['label'] = meta_df['filename'].apply(extract_label)
    
    plt.figure(figsize=(8, 5))
    sns.countplot(data=meta_df, x='label', hue='label', palette=['#E74C3C', '#2ECC71', '#95A5A6'], legend=False)
    plt.title('Dataset Class Balance: Truth vs. Lie')
    plt.xlabel('Ground Truth Label')
    plt.ylabel('Number of Video Clips')
    plt.show()
    
    print("Class Distribution:")
    print(meta_df['label'].value_counts())

## OpenFace Quality Analysis and Feature Extraction
Here, we initialize the `VisualFeatureExtractor` to run OpenFace on a sample of the videos. The block extracts the facial features, evaluates tracking confidence, and calculates the Root Mean Square (RMS) jitter for spatial translation (Tx, Ty, Tz). High dropout rates or excessive jitter indicate poor tracking stability, which can negatively impact Action Unit recognition.

In [ ]:
JITTER_THRESH = 5.0
CONF_THRESH = 0.75


extractor = VisualFeatureExtractor(openface_bin_path=OPENFACE_BIN, device=DEVICE)
results = []

# Test up to 10 randomly selected videos
for _, r in meta_df.sample(min(10, len(meta_df)), random_state=42).iterrows():
    path, name = r["path"], os.path.basename(r["path"])
    cap_meta = cv2.VideoCapture(path)
    reported_fps = cap_meta.get(cv2.CAP_PROP_FPS)
    video_frame_count = cap_meta.get(cv2.CAP_PROP_FRAME_COUNT)
    cap_meta.release()

    # Detect scene boundaries; use the entire video if no cuts are detected
    scenes = [(s.get_frames() + 1, e.get_frames())
              for s, e in detect(path, ContentDetector())] or [(1, float("inf"))]

    # Extract facial landmarks, head pose, gaze, and tracking confidence
    with tempfile.TemporaryDirectory() as tmp:
        df = extractor.run_openface(path, tmp, landmarks=False)
    if df is None or df.empty:
        continue
    df.columns = df.columns.str.strip()

    for i, (start, end) in enumerate(scenes, 1):
        scene = df[df.frame.between(start, end)].copy()
        if scene.empty:
            continue

        # Exclude low-confidence frames from motion and gaze calculations
        valid = scene[scene.confidence >= CONF_THRESH]
        pos = ["pose_Tx", "pose_Ty", "pose_Tz"]
        rot = ["pose_Rx", "pose_Ry", "pose_Rz"]
        gaze = ["gaze_angle_x", "gaze_angle_y"]

        # Estimate frame-to-frame head translation jitter (consecutive frames only)
        diff = valid[pos].diff().where(valid.frame.diff().eq(1))
        jitter = np.sqrt(diff.pow(2).mean()).fillna(0).values

        # Calculate head rotation and gaze variability in degrees
        head_std = valid[rot].std().fillna(0).values * 180 / np.pi
        gaze_std = valid[gaze].std().fillna(0).values * 180 / np.pi
        dropout = (scene.confidence < CONF_THRESH).mean() * 100

        # --- FPS consistency and bounding-box stability ---
        fps_measured = np.nan
        fps_delta_pct = np.nan
        if "timestamp" in scene.columns:
            dt = scene["timestamp"].diff()
            frame_diff = scene["frame"].diff()
            rates = (frame_diff / dt).replace([np.inf, -np.inf], np.nan)
            rates = rates[(dt > 0) & (frame_diff > 0)]
            if not rates.empty:
                fps_measured = float(rates.median())
                if reported_fps and reported_fps > 0:
                    fps_delta_pct = abs(fps_measured - reported_fps) / reported_fps * 100

        bbox_center_jitter_px = np.nan
        bbox_size_jitter_px = np.nan
        x_cols = [f"x_{j}" for j in range(68)]
        y_cols = [f"y_{j}" for j in range(68)]
        if all(c in valid.columns for c in x_cols + y_cols) and len(valid) >= 2:
            x = valid[x_cols].to_numpy(dtype=float)
            y = valid[y_cols].to_numpy(dtype=float)
            finite_rows = np.isfinite(x).all(axis=1) & np.isfinite(y).all(axis=1)
            x, y = x[finite_rows], y[finite_rows]
            frame_ids = valid.loc[finite_rows, "frame"].to_numpy()

            if len(x) >= 2:
                left, right = x.min(axis=1), x.max(axis=1)
                top, bottom = y.min(axis=1), y.max(axis=1)
                centers = np.column_stack(((left + right) / 2, (top + bottom) / 2))
                sizes = np.column_stack((right - left, bottom - top))
                consecutive = np.diff(frame_ids) == 1
                if consecutive.any():
                    bbox_center_jitter_px = float(
                        np.linalg.norm(np.diff(centers, axis=0)[consecutive], axis=1).mean()
                    )
                    bbox_size_jitter_px = float(
                        np.abs(np.diff(sizes, axis=0)[consecutive]).mean()
                    )

        results.append({
            "video": f"{name}_scene_{i}", "dropout": dropout,
            "reported_fps": reported_fps, "measured_fps": fps_measured,
            "fps_delta_pct": fps_delta_pct,
            "bbox_center_jitter_px": bbox_center_jitter_px,
            "bbox_size_jitter_px": bbox_size_jitter_px,
            **dict(zip(["j_tx", "j_ty", "j_tz"], jitter)),
            **dict(zip(["head_pitch_std", "head_yaw_std", "head_roll_std"], head_std)),
            **dict(zip(["gaze_x_std", "gaze_y_std"], gaze_std))
        })
        print(
            f"{name}_scene_{i} | Dropout: {dropout:.1f}% | "
            f"Head jitter: {jitter.round(2)} | "
            f"FPS: {reported_fps:.2f}/{fps_measured:.2f} | "
            f"BBox center jitter: {bbox_center_jitter_px:.2f}px"
        )

        # Play scenes with high jitter and overlay landmarks and gaze directions
        # Play original and debug videos side by side for high-jitter scenes
        if False and jitter.max() > JITTER_THRESH:
            cap = cv2.VideoCapture(path)
            cap.set(cv2.CAP_PROP_POS_FRAMES, start - 1)
            delay = int(1000 / (cap.get(cv2.CAP_PROP_FPS) or 30))
            x_cols, y_cols = [f"x_{j}" for j in range(68)], [f"y_{j}" for j in range(68)]

            while cap.isOpened():
                ret, original = cap.read()
                if not ret:
                    break

                idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
                if idx > end:
                    break

                debug = original.copy()
                d = scene[scene.frame == idx]

                if not d.empty:
                    d = d.iloc[0]
                    color = (0, 255, 0) if d.confidence >= CONF_THRESH else (0, 0, 255)

                    # Draw facial landmarks
                    if all(c in scene.columns for c in x_cols + y_cols):
                        for x, y in zip(d[x_cols], d[y_cols]):
                            if pd.notna(x) and pd.notna(y):
                                cv2.circle(debug, (int(x), int(y)), 3, color, -1)

                    # Draw gaze vectors for confident detections
                    if d.confidence >= CONF_THRESH:
                        dx = int(100 * np.sin(d.gaze_angle_x))
                        dy = int(-100 * np.sin(d.gaze_angle_y))

                        for ids in [range(36, 42), range(42, 48)]:
                            eye = (int(d[[f"x_{j}" for j in ids]].mean()),
                                int(d[[f"y_{j}" for j in ids]].mean()))
                            cv2.arrowedLine(debug, eye, (eye[0] + dx, eye[1] + dy),
                                            (255, 255, 0), 2, tipLength=0.2)

                        cv2.putText(debug, f"Gaze X: {np.degrees(d.gaze_angle_x):+.1f} deg",
                                    (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

                # Label both panes
                cv2.putText(original, "Original Video", (20, 35),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                cv2.putText(debug, f"Debug | Scene {i} | q: skip", (20, 35),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

                # Resize both panes equally and display side by side
                h = min(original.shape[0], debug.shape[0])
                w = int(original.shape[1] * h / original.shape[0])
                original = cv2.resize(original, (w, h))
                debug = cv2.resize(debug, (w, h))
                cv2.imshow("Original vs Debug", cv2.hconcat([original, debug]))

                if cv2.waitKey(delay) & 0xFF == ord("q"):
                    break

            cap.release()
            cv2.destroyAllWindows()

# Summarize average jitter and gaze dispersion across processed scenes
if results:
    res = pd.DataFrame(results)
    print("\n--- Dataset Averages ---")
    print("Mean Jitter (Tx, Ty, Tz):",
          res[["j_tx", "j_ty", "j_tz"]].mean().round(2).values)
    print(f"Mean Gaze Dispersion: X={res.gaze_x_std.mean():.2f}° | "
          f"Y={res.gaze_y_std.mean():.2f}°")

## Emotion Features — Excluded

The previous emotion-probability analysis was based on randomly generated mock values and an untrained placeholder model. These values are not valid measurements and have been excluded. The current visual extractor uses 49 OpenFace features plus 2 quality features (51 total). Emotion features can be reconsidered when a trained and validated model is available.

### FPS and Bounding-Box Stability
The EDA records reported video FPS and, when OpenFace timestamps are available, measured FPS and percentage difference. Face bounding boxes are approximated from the 68 facial landmarks; center displacement and box-size change are calculated between consecutive confident frames. These are tracking-quality diagnostics, not model features.
